# 14 — Gold DLT: Fact Tables

## Configuration

In [0]:
# fact_street_readings, fact_traffic_counts, fact_citizen_reports.
#
# Deliberately batch reads (spark.table), not streaming — unlike the
# dimension SCD2 tables in 13_gold_dimensions.py, these don't need
# apply_changes(), so there's no reason to force a streaming source. Each
# fact is a full, correct recomputation of its grain on every pipeline run.
# fact_street_readings' source (streets_business) is a plain Delta table
# that gets both a full overwrite (11_silver_business_rules.py) and a
# manual UPDATE (12_delta_acid_timetravel_demo.py) — reading it as a stream
# would require ignoreChanges/ignoreDeletes and still only give a point-in-
# time snapshot anyway, so batch is both simpler and more correct here.

import dlt
from pyspark.sql import functions as F

CATALOG = spark.conf.get("pipeline.catalog", "vstone_catalog")
SILVER = f"{CATALOG}.{spark.conf.get('pipeline.silver_schema', 'silver')}"

GOLD_PROPS = {
    "quality": "gold",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}

# Fact Street Readings 
 Grain: (street_id, reading_ts). FK: street_id -> dim_street, reading_date -> dim_date. Reads raining_clipped from streets_business (Day 5's
 business-rule table), not raw streets_silver.

In [0]:
@dlt.table(
    name="fact_street_readings",
    comment="Gold Fact: street sensor readings, business rules applied (raining clipped). "
            "Grain: (street_id, reading_ts). "
            "FK street_id -> dim_street.street_id. FK reading_date -> dim_date.date_key. "
            "AUDIT: bronze_load_dt -> silver_load_dt -> business_load_dt -> gold_load_dt.",
    table_properties={
        **GOLD_PROPS, "type": "fact", "grain": "street_id,reading_ts",
        "fk_street": "street_id -> dim_street.street_id",
        "fk_date": "reading_date -> dim_date.date_key",
    },
)
@dlt.expect("valid_street_fk", "street_id IS NOT NULL")
@dlt.expect("valid_date_fk", "reading_date IS NOT NULL")
def fact_street_readings():
    return (
        spark.table(f"{SILVER}.streets_business")
        .select(
            "street_id",
            F.to_date("reading_ts").alias("reading_date"),
            "reading_ts", "reading_hour",
            "noise", "pollution", "light",
            "raining", "raining_clipped",
            "bronze_load_dt", "bronze_source", "silver_load_dt", "business_load_dt",
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )


## Fact Traffic Counts
 Grain: (location, reading_ts). FK: location -> dim_node_location, reading_date -> dim_date.

In [0]:
@dlt.table(
    name="fact_traffic_counts",
    comment="Gold Fact: traffic sensor vehicle counts (enter_count/exit_count confirmed as "
            "counts, not hour-of-day values, via Day 1 profiling). Grain: (location, reading_ts). "
            "FK location -> dim_node_location.location. FK reading_date -> dim_date.date_key. "
            "AUDIT: bronze_load_dt -> silver_load_dt -> gold_load_dt.",
    table_properties={
        **GOLD_PROPS, "type": "fact", "grain": "location,reading_ts",
        "fk_location": "location -> dim_node_location.location",
        "fk_date": "reading_date -> dim_date.date_key",
    },
)
@dlt.expect("valid_location_fk", "location IS NOT NULL")
@dlt.expect("valid_date_fk", "reading_date IS NOT NULL")
def fact_traffic_counts():
    return (
        spark.table(f"{SILVER}.cars_silver")
        .select(
            "location",
            F.to_date("reading_ts").alias("reading_date"),
            "reading_ts", "enter_count", "exit_count",
            "bronze_load_dt", "bronze_source", "silver_load_dt",
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )

## Fact Citizen Reports

In [0]:
# Grain: (message, message_date, message_hour) — no id column exists.
# street_id is a BEST-EFFORT FK: resolved by checking whether the cleaned
# message text contains a street's exact name, not an authoritative key —
# a message may mention no street (street_id NULL) or, rarely, more than
# one (only the first match by street_id is kept, arbitrarily — flagged,
# not silently precise). streets_list_silver only has 36 rows, so a
# broadcast cross-join-and-filter is cheap even though it's not an equi-join.

@dlt.table(
    name="fact_citizen_reports",
    comment="Gold Fact: citizen traffic/incident reports. Grain: (message, message_date, message_hour) "
            "— no id column exists in the source. FK message_date -> dim_date.date_key. "
            "FK street_id -> dim_street.street_id is BEST-EFFORT (text-contains match on street name, "
            "not an authoritative key) — NULL where no street name matched. "
            "AUDIT: silver_load_dt -> gold_load_dt.",
    table_properties={
        **GOLD_PROPS, "type": "fact", "grain": "message,message_date,message_hour",
        "fk_date": "message_date -> dim_date.date_key",
        "fk_street": "street_id -> dim_street.street_id (best-effort text match, nullable)",
    },
)
@dlt.expect("valid_date_fk", "message_date IS NOT NULL")
def fact_citizen_reports():
    telegram = (
        spark.table(f"{SILVER}.telegram_silver")
        .withColumn("message_date", F.to_date(F.col("message_date"), "dd/MM/yyyy"))
        .withColumn("row_id", F.monotonically_increasing_id())
    )

    streets = (
        dlt.read("dim_street")
        .filter(F.col("__END_AT").isNull())
        .select("street_id", "street_name")
    )

    matched = (
        telegram
        .crossJoin(F.broadcast(streets))
        .filter(F.col("message").contains(F.col("street_name")))
    )
    # A message could theoretically contain more than one street name —
    # keep one match per row_id, arbitrary tie-break, not claimed to be
    # the "correct" one when ambiguous.
    from pyspark.sql.window import Window
    w = Window.partitionBy("row_id").orderBy("street_id")
    best_match = (
        matched
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .select("row_id", "street_id")
    )

    return (
        telegram
        .join(best_match, on="row_id", how="left")
        .select(
            "message", "message_date", "message_hour", "street_id",
            "bronze_load_dt", "bronze_source", "silver_load_dt",
        )
        .withColumn("gold_load_dt", F.current_timestamp())
    )


In [0]:
display(
    spark.table("vstone_catalog.gold.fact_traffic_counts")
    .groupBy("location")
    .count()
    .orderBy("location")
)

In [0]:
for table in ["vstone_catalog.bronze.cars_dlt",
              "vstone_catalog.silver.cars_silver",
              "vstone_catalog.gold.fact_traffic_counts"]:
    count = spark.sql(f"SELECT COUNT(*) c FROM {table} WHERE location = 7").collect()[0]["c"]
    print(f"{table}: {count:,} rows reference location=7")

In [0]:
quarantined = spark.sql(
    "SELECT COUNT(*) c FROM vstone_catalog.silver.cars_silver_quarantine WHERE location = 7"
).collect()[0]["c"]
print(f"location=7 rows in quarantine: {quarantined:,}")

In [0]:
from pyspark.sql import functions as F

bronze = (
    spark.table("vstone_catalog.bronze.cars_dlt")
    .filter("location = 7")
)

# Duplicates on the grain only (location, date)
grain_dupes = (
    bronze
    .groupBy("location", "date")
    .count()
    .filter("count > 1")
)

grain_dupe_rows = (
    grain_dupes
    .agg(F.coalesce(F.sum("count"), F.lit(0)).alias("total"))
    .collect()[0]["total"]
    - grain_dupes.count()
)

# Duplicates on the FULL row
full_row_dupes = (
    bronze
    .groupBy(*bronze.columns)
    .count()
    .filter("count > 1")
)

full_dupe_rows = (
    full_row_dupes
    .agg(F.coalesce(F.sum("count"), F.lit(0)).alias("total"))
    .collect()[0]["total"]
    - full_row_dupes.count()
)

print(f"Rows lost to grain-only dedup: {grain_dupe_rows:,}")
print(f"Rows that are TRUE full-row duplicates: {full_dupe_rows:,}")
print(
    f"Rows with SAME timestamp but DIFFERENT enter/exit "
    f"(real data at risk): {grain_dupe_rows - full_dupe_rows:,}"
)

In [0]:
bronze7 = spark.table("vstone_catalog.bronze.cars_dlt").filter("location = 7")

triple_key_dupes = bronze7.groupBy("location", "date").count().filter("count > 1")
triple_dupe_rows = triple_key_dupes.agg({"count": "sum"}).collect()[0][0] or 0
print(f"Rows still duplicated on (location, date, id): {triple_dupe_rows:,}")

In [0]:
from pyspark.sql import functions as F

dupe_grain = (bronze7.groupBy("location", "date")
              .count().filter("count > 1")
              .select("location", "date").limit(3))

bronze7.join(dupe_grain, on=["location", "date"], how="inner") \
       .orderBy("date") \
       .show(20, truncate=False)